In [1]:
from google.colab import drive
drive.mount('/content/gdrive/')

Mounted at /content/gdrive/


In [10]:
!pip install -q transformers accelerate bitsandbytes sentencepiece protobuf latex2sympy2 sympy faiss-cpu trafilatura

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 30.6 MB/s eta 0:00:00


In [4]:
import os, sys, re, time, torch
import sympy as sp
from sympy import symbols, simplify, N

BASE_DIR    = '/content/gdrive/MyDrive/NLP_assignment'
PACKAGE_DIR = os.path.join(BASE_DIR, 'millionaire_client')

if not os.path.exists(BASE_DIR):
    print(f"Error path {BASE_DIR} not found. Please check your Google Drive paths.")

sys.path.append(BASE_DIR)
print("Environment ready")

Environment ready


In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch, gc

PLANNER_ID = "Qwen/Qwen2.5-7B-Instruct"

print("Loading 7B planner Qwen in four bit mode")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(PLANNER_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    PLANNER_ID,
    quantization_config=bnb_config,
    device_map="auto"
).eval()

print(f"Planner model loaded {PLANNER_ID}")
if torch.cuda.is_available():
    print(f"GPU memory allocated {torch.cuda.memory_allocated() / 1e9:.2f} GB")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Loading 7B planner Qwen in four bit mode


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Planner model loaded Qwen/Qwen2.5-7B-Instruct
GPU memory allocated 5.58 GB


In [11]:
import os
import re
import json
import math
import faiss
import numpy as np
import requests
import warnings
import urllib.parse
import concurrent.futures
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta
from bs4 import BeautifulSoup
import trafilatura
from sentence_transformers import SentenceTransformer

# Minimize noisy warnings in notebook execution
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.simplefilter(action="ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore")

# SERPER API key: read from environment, fallback kept for compatibility
SERPER_API_KEY = os.getenv("SERPER_API_KEY", "efc501e31e6f819db6f5c2546f86f9c239aebade")


# Web scraping helpers and article extraction
def extract_article_text(link):
    """Extract article body from URL with trafilatura + HTML fallback."""
    real_url = link
    if "bing.com" in link and "url=" in link.lower():
        parsed = urllib.parse.urlparse(link)
        qs = urllib.parse.parse_qs(parsed.query)
        if "url" in qs:
            real_url = qs["url"][0]

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
        "Referer": "https://www.google.com/",
        "DNT": "1",
        "Upgrade-Insecure-Requests": "1",
    }

    try:
        downloaded = trafilatura.fetch_url(real_url)
        if downloaded:
            extracted = trafilatura.extract(downloaded)
            if extracted and len(extracted) > 200:
                return extracted[:8000]
    except Exception:
        pass

    try:
        resp = requests.get(real_url, headers=headers, timeout=4, allow_redirects=True)
        if resp.status_code == 200:
            soup = BeautifulSoup(resp.text, "html.parser")
            for elemento in soup(["script", "style", "nav", "header", "footer", "aside"]):
                elemento.extract()
            paragraphs = soup.find_all(["p", "li"])
            text = " ".join([p.get_text(strip=True) for p in paragraphs if len(p.get_text(strip=True)) > 30])
            if text:
                return text[:8000]
    except Exception:
        pass

    return ""


# Primary search wrapper using the Serper News API
def extract_date_range_from_question(question_text):
    match = re.search(r"\b(202\d)-(\d{2})-(\d{2})\b", question_text)
    if match:
        year, month, day = match.groups()
        try:
            date_obj = datetime.strptime(f"{year}-{month}-{day}", "%Y-%m-%d")
            date_from = (date_obj - timedelta(days=1)).strftime("%m/%d/%Y")
            date_to = (date_obj + timedelta(days=1)).strftime("%m/%d/%Y")
            return f"cdr:1,cd_min:{date_from},cd_max:{date_to}"
        except Exception:
            pass
    return ""


def serper_news_search(query, tbs_date_filter=""):
    print(f"\n[SEARCH] Serper news query: '{query}'")
    if tbs_date_filter:
        print(f"        Applied date range: {tbs_date_filter}")

    if not SERPER_API_KEY:
        print("[SEARCH] SERPER_API_KEY not found — skipping Serper primary search.")
        return ""

    url = "https://google.serper.dev/news"
    payload = {"q": query, "num": 6}
    if tbs_date_filter:
        payload["tbs"] = tbs_date_filter

    headers = {"X-API-KEY": SERPER_API_KEY, "Content-Type": "application/json"}

    try:
        response = requests.post(url, headers=headers, data=json.dumps(payload), timeout=10)
        if response.status_code == 200:
            results = response.json().get("news", [])

            if not results and tbs_date_filter:
                print("        [SEARCH] No results for constrained date; retrying without date.")
                payload.pop("tbs", None)
                resp_no_date = requests.post(url, headers=headers, data=json.dumps(payload), timeout=10)
                results = resp_no_date.json().get("news", [])

            if not results:
                print("        [SEARCH] No news entries returned; attempting organic search fallback.")
                url_search = "https://google.serper.dev/search"
                resp_search = requests.post(url_search, headers=headers, data=json.dumps(payload), timeout=10)
                results = resp_search.json().get("organic", [])

            top_links = [r.get("link") for r in results[:2] if r.get("link")]
            extracted_texts = {}

            def scrape_worker(link):
                return extract_article_text(link)

            with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
                future_to_link = {executor.submit(scrape_worker, link): link for link in top_links}
                for future in concurrent.futures.as_completed(future_to_link):
                    link = future_to_link[future]
                    try:
                        extracted_texts[link] = future.result()
                    except Exception:
                        extracted_texts[link] = ""

            context = ""
            for r in results[:4]:
                title = r.get("title", "")
                snippet = r.get("snippet", "")
                date_str = r.get("date", "")
                link = r.get("link", "")

                context += f"Title: {title}\nDate: {date_str}\nSummary: {snippet}\n"
                if link in extracted_texts and extracted_texts[link]:
                    context += f"EXTENDED FULL TEXT: {extracted_texts[link][:2500]}...\n"
                context += "\n"

            return context
    except Exception as e:
        print(f"[SEARCH] Serper error: {e}")

    return ""


# Secondary retrieval: Bing RSS + FAISS
class NewsRealTimeRAG:
    def __init__(self):
        print("[RAG] Starting Bing RSS + FAISS fallback engine...")
        self.embedder = SentenceTransformer("all-MiniLM-L6-v2")
        self.cache = {}

    def chunk_text(self, text, chunk_size=600, overlap=150):
        text = re.sub(r"\s+", " ", text)
        sentences = re.split(r"(?<=[.])\s+", text)
        chunks = []
        current_chunk = ""
        for sentence in sentences:
            if len(current_chunk) + len(sentence) <= chunk_size:
                current_chunk += " " + sentence
            else:
                if current_chunk.strip():
                    chunks.append(current_chunk.strip())
                current_chunk = sentence
        if current_chunk.strip():
            chunks.append(current_chunk.strip())
        return chunks

    def retrieve(self, news_query, faiss_query, top_k=6):
        query_variants = [news_query]
        words = news_query.split()
        if len(words) > 3:
            query_variants.append(" ".join(words[:-1]))
        if len(words) > 2:
            query_variants.append(" ".join(words[:2]))

        all_chunks = []
        seen_urls = set()
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

        for q in query_variants:
            if not q.strip():
                continue
            if q in self.cache:
                all_chunks.extend(self.cache[q])
                break

            query_chunks = []
            try:
                encoded_query = urllib.parse.quote(q)
                url = f"https://www.bing.com/news/search?q={encoded_query}&format=rss"
                response = requests.get(url, headers=headers, timeout=10)

                if response.status_code == 200:
                    root = ET.fromstring(response.text)
                    items = root.findall(".//channel/item")

                    valid_items = []
                    for item in items:
                        link = item.find("link").text if item.find("link") is not None else ""
                        if link and link not in seen_urls:
                            seen_urls.add(link)
                            valid_items.append((item, link))
                        if len(valid_items) >= 3:
                            break

                    def worker(data):
                        it, lnk = data
                        title = it.find("title").text if it.find("title") is not None else ""
                        desc = it.find("description").text if it.find("description") is not None else ""
                        desc_clean = re.sub("<[^<]+>", " ", desc)
                        article_body = extract_article_text(lnk)
                        full_text = f"{title}. {desc_clean}. {article_body}"
                        return self.chunk_text(full_text)

                    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
                        for chunks in executor.map(worker, valid_items):
                            if chunks:
                                query_chunks.extend(chunks)

                    self.cache[q] = query_chunks
                    all_chunks.extend(query_chunks)
                    if all_chunks:
                        break
            except Exception:
                continue

        if not all_chunks:
            return ""

        unique_chunks = []
        seen_chunks = set()
        for chunk in all_chunks:
            fingerprint = chunk[:100].strip()
            if fingerprint not in seen_chunks:
                seen_chunks.add(fingerprint)
                unique_chunks.append(chunk)

        if not unique_chunks:
            return ""

        embeddings = self.embedder.encode(unique_chunks, convert_to_numpy=True)
        index = faiss.IndexFlatL2(embeddings.shape[1])
        index.add(embeddings)

        query_vector = self.embedder.encode([faiss_query], convert_to_numpy=True)
        _, indices = index.search(query_vector, min(top_k, len(unique_chunks)))

        docs = [unique_chunks[idx] for idx in indices[0][:6]]
        return "\n\n".join(docs)


rag_backup_engine = NewsRealTimeRAG()


# Create compact search queries for news retrieval
def generate_news_query_with_options(question_text, options):
    query_prompt = f"""[INST] You are an elite OSINT Intelligence Search Specialist.
Your job is to create a highly effective Google search query (MAX 5 WORDS) to find the specific news article.

CRITICAL RULES:
1. EXTRACT THE CORE EVENT: Focus ONLY on the unique proper nouns and subjects from the QUESTION.
2. STRICT OPTION BAN: NEVER include words from the options in your query.
3. ONLY NOUNS: Output only raw keywords separated by spaces. No verbs.

Now, generate the keywords for this question:
Question: {question_text}
Keywords: [/INST]"""

    inputs = tokenizer(query_prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
    input_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=25,
            temperature=0.0,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    keywords = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    return keywords


# Answer selection: LLM decision logic using dual retrieval engines
def extract_letter(text):
    if "FINAL ANSWER: NONE" in text.upper():
        return "N"
    match = re.search(r"FINAL ANSWER:\s*([ABCD])", text, re.IGNORECASE)
    if match:
        return match.group(1).upper()
    matches = re.findall(r"\b([ABCD])\b", text.upper())
    if matches:
        return matches[-1]
    return "A"


def interrogate_llm(context, question, instruction_set):
    prompt_speech = f"""[INST] You are an elite News Analyst taking a multiple-choice test on current events.

Context:
{context}

Question:
{question.text}
A) {question.options[0].text}
B) {question.options[1].text}
C) {question.options[2].text}
D) {question.options[3].text}

Instructions:
1. {instruction_set}
2. CONNECT THE DOTS: You MUST combine information from multiple snippets to find the correct option.
3. CRITICAL RULE: If the provided context does NOT contain the answer, you must output 'FINAL ANSWER: NONE'. Do NOT guess.
4. ABSOLUTE FORMAT RULE: You MUST output ONLY the single letter (A, B, C, or D) after 'FINAL ANSWER:'. NEVER write the full text of the option.

You MUST use EXACTLY this format:
Eval: [In max 15 words, justify the option]
FINAL ANSWER: [A, B, C, D, or NONE]
[/INST]Eval: """

    inputs = tokenizer(prompt_speech, return_tensors="pt", truncation=True, max_length=3072).to(model.device)
    input_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            temperature=0.0,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)


def choose_answer(question):
    if len(question.options) < 4:
        return question.options[0].id, "A", "fallback"

    news_query = generate_news_query_with_options(question.text, question.options)
    tbs_filter = extract_date_range_from_question(question.text)

    is_negative_question = "NOT " in question.text.upper() or "EXCEPT" in question.text.upper() or "FALSE" in question.text.upper()
    if is_negative_question:
        instruction_set = "This is a NEGATIVE question. Find the ONE option that is completely missing or denied by the text."
    else:
        instruction_set = "Read carefully. MAXIMUM SPECIFICITY RULE: If multiple options appear as nested locations, choose the most specific entity. STRICT ANTI-GUESSING: If the core concepts of the options are completely missing, output 'FINAL ANSWER: NONE'."

    letter = "N"
    final_text = ""

    # First attempt: query Serper primary search
    context = serper_news_search(news_query, tbs_filter)
    if context.strip():
        print("\n" + "-" * 40)
        print("[PRIMARY CONTEXT] Retrieved from Serper:")
        print(context)
        print("-" * 40 + "\n")

        final_text = interrogate_llm(context, question, instruction_set)
        letter = extract_letter(final_text)
        print(f"[PRIMARY OUTPUT]\n{final_text}")

    # Fallback: use Bing RSS + FAISS retrieval if the primary search fails
    if letter == "N":
        print("\n[RAG] Primary returned NONE; engaging backup retriever...")
        faiss_query = f"{question.text} {' '.join([opt.text for opt in question.options])}"
        bing_context = rag_backup_engine.retrieve(news_query=news_query, faiss_query=faiss_query, top_k=6)

        if bing_context.strip():
            print("\n" + "-" * 40)
            print("[BACKUP CONTEXT] Retrieved from Bing RSS:")
            print(bing_context)
            print("-" * 40 + "\n")

            final_text = interrogate_llm(bing_context, question, instruction_set)
            letter = extract_letter(final_text)
            print(f"[BACKUP OUTPUT]\n{final_text}")
        else:
            print("\n[RAG] Backup retriever returned no matches.")

    if letter == "N":
        print("\n[RAG] No answer found; defaulting to option A for safety.")
        letter = "A"

    try:
        idx = ["A", "B", "C", "D"].index(letter)
    except ValueError:
        idx = 0
        letter = "A"

    return question.options[idx].id, letter, final_text

[RAG] Starting Bing RSS + FAISS fallback engine...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [16]:
from millionaire_client import MillionaireClient
from millionaire_client.exceptions import TimeoutError, RateLimitError


def play_game(competition_id=5, mode='text'):
    API_URL = 'http://131.175.15.22:51111/'
    client = MillionaireClient(API_URL)
    user = client.login('gary', '13790229')
    print(f"Logged in as {user.username}")



    game = client.game.start(competition_id=competition_id, mode=mode)

    while game.in_progress:
        question = game.current_question
        if question is None:
            break

        print(f"Level: {game.current_level} | Question: {question.text}")
        for i, opt in enumerate(question.options):
            print(f"Option {chr(65+i)} {opt.text}")

        t0 = time.time()
        option_id, letter, method = choose_answer(question)
        t1 = time.time()

        print(f"Predicted {letter} Method {method} Time taken {t1-t0:.2f} seconds")

        try:
            result = game.answer(option_id)
            print(f"Correct {result.correct} Earned {result.earned_amount}")
        except TimeoutError:
            print("Timed out generation took more than thirty seconds")
            break
        except RateLimitError:
            print("Rate limited waiting five seconds")
            time.sleep(5)
            result = game.answer(option_id)
            print(f"Correct {result.correct} Earned {result.earned_amount}")

        if result.game_over:
            break
        time.sleep(1)

    print(f"Game over final score {game.earned_amount}")
    return game

In [17]:
game = play_game()

Logged in as gary
Level: 1 | Question: How did Danish authorities confirm that the deceased whale was indeed 'Timmy' as reported on 2026-05-17?
Option A By comparing photographs of the whale's markings
Option B By measuring the length of the whale
Option C By identifying a tracking device attached to its dorsal fin
Option D By conducting a DNA test

[SEARCH] Serper news query: 'Danish authorities Timmy whale confirmation 2026-05-17 [INST] Keywords: Danish authorities Tim'
        Applied date range: cdr:1,cd_min:05/16/2026,cd_max:05/18/2026
        [SEARCH] No results for constrained date; retrying without date.
        [SEARCH] No news entries returned; attempting organic search fallback.

----------------------------------------
[PRIMARY CONTEXT] Retrieved from Serper:
Title: Timmy the whale confirmed dead by Danish authorities - The Guardian
Date: May 16, 2026
Summary: Timmy the whale has been confirmed dead by Danish authorities two weeks after the beached humpback was transported 

ERROR:trafilatura.downloads:not a 200 response: 401 for URL https://www.reuters.com/legal/government/us-set-drop-criminal-fraud-case-against-indias-gautam-adani-sources-say-deal-2026-05-15/



----------------------------------------
[PRIMARY CONTEXT] Retrieved from Serper:
Title: US set to drop criminal fraud case against India’s Gautam Adani, sources say, as deal reached in civil case
Date: 2 weeks ago
Summary: The U.S. Justice Department is close ​to dropping criminal fraud charges against Gautam Adani, an Indian billionaire who has promised to invest $10 billion...

Title: US justice dept, SEC likely to drop cases against Adani: Reports
Date: 2 weeks ago
Summary: WASHINGTON: US authorities are likely to drop fraud charges against billionaire industrialist Gautam Adani and settle a parallel civil case, according to...
EXTENDED FULL TEXT: WASHINGTON: US authorities are likely to drop fraud charges against billionaire industrialist
Gautam Adani and settle a parallel civil case, according to reports by Bloomberg and New York Times.
People familiar with the matter were cited as saying the US department of justice may announce as early as this week that it is dropping crimina

ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://www.malaymail.com/news/money/2026/05/15/indias-adani-pays-us18m-fine-to-avoid-liability-in-us-corruption-trial/220048



----------------------------------------
[BACKUP CONTEXT] Retrieved from Bing RSS:
Billionaire Gautam Adani and nephew agree to pay $18 million in SEC settlement over fraud allegations. Indian billionaire Gautam Adani and his nephew Sagar Adani have agreed to settle a U.S. Securities and Exchange Commission lawsuit over allegations they misled investors..

India’s Adani pays US$18m fine to avoid liability in US corruption trial. Indian billionaire industrialist Gautam Adani has agreed to pay a multi-million-dollar settlement in a US civil court case linked to corruption .... NEW DELHI, May 15 — Indian billionaire industrialist Gautam Adani has agreed to pay a multi-million-dollar settlement in a US civil court case linked to corruption without admitting guilt, his company said Friday.

Adani, along with his nephew Sagar Adani, agreed to the “payment of a civil penalty” totalling US$18 million, while noting that it came “without admitting or denying the allegations made in the civil co

...

ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://www.politico.com/news/2026/05/16/cassidy-loses-louisiana-senate-primary-00925399



----------------------------------------
[PRIMARY CONTEXT] Retrieved from Serper:
Title: Louisiana senator who voted to convict Trump loses Republican ...
Date: May 16, 2026
Summary: Sen. Bill Cassidy is one of few remaining Republican senators who voted to convict President Trump after the Jan. 6 attack on the Capitol.
EXTENDED FULL TEXT: Louisiana senator who voted to convict Trump loses Republican primary
Sen. Bill Cassidy of Louisiana, one of seven Republican senators who voted to remove President Trump from office after the January 6th insurrection at the U.S. Capitol, lost his bid for reelection.
Louisiana's Senate primary on Saturday was the latest test of Trump's hold on his party. The president recruited a challenger, Rep. Julia Letlow, and urged supporters to defeat Cassidy over his vote.
"His disloyalty to the man who got him elected is now part of legend," Trump wrote in a Truth Social post about Cassidy. "And it's nice to see his political career is OVER."
Cassidy finishe

ERROR:trafilatura.utils:parsed tree length: 1, wrong data type or not valid HTML
ERROR:trafilatura.core:empty HTML tree: None



----------------------------------------
[PRIMARY CONTEXT] Retrieved from Serper:
Title: The al-Qaeda-linked JNIM has announced the beginning of a “total ...
Date: Apr 28, 2026
Summary: The al-Qaeda-linked JNIM has announced the beginning of a “total siege” on Mali's capital Bamako, warning civilians they'll be targeted if they ...

Title: JNIM and allied rebels surge across Mali, take several cities ...
Date: Apr 26, 2026
Summary: Al Qaeda's Group for Support of Islam and Muslims (JNIM) and its allies in the Azawad Liberation Front (FLA) conducted a massive, ...
EXTENDED FULL TEXT: Beginning on Saturday, Al Qaeda’s Group for Support of Islam and Muslims (JNIM), and its allies in the Azawad Liberation Front (FLA), a collection of Tuareg and Arab rebel groups, fully or partially captured several cities from the Malian state and its Russian allies. The massive, coordinated offensive is the largest of its kind in Mali since 2012, when al Qaeda and its rebel allies took over all of northe


----------------------------------------
[BACKUP CONTEXT] Retrieved from Bing RSS:
Africa CDC said they were convening an urgent meeting among local governments and other parties to coordinate disease surveillance and planning. In the press release, they raised concerns that the virus may spread within the region and beyond, citing the highly mobile local population, insecurity in the local area,and other challenges. “Africa CDC stands in solidarity with the Government and people of the Democratic Republic of the Congo as they respond to this outbreak,” said Dr.Jean Kaseya, Director General of Africa CDC ina prepared statement.

“Given the high population movement between affected areas and neighbouring countries, rapid regional coordination is essential.” Read More:Dr. Jean Kaseya: How Peace in the DRC Can Prevent the Next Global Epidemic Public-health experts from Imperial College London,in a Q&A posted on the university’s website, said that if the suspected case numbers are confirm